# Example Progression 19 — Approvals: a run that pauses for a person (SDK, interactive)

Interactive counterpart of [`wf_examples/wf_example_progression_19.py`](../../wf_examples/wf_example_progression_19.py).
It imports the **exact same** `build_assistant_workflow()` from that script (no duplication), uploads it, starts it as
a **durable background run**, approves the step waiting for a person, follows the run to the end, and cleans up.

The Pause Workflow node is enabled per server (`CONTROL_WAIT_NODES_ENABLED`). Where it is off, the upload is refused
and the notebook says so and stops cleanly. No LLM is called.

> **Async-first.** Uses `AsyncWorkflowClient` with top-level `await` (the setup cell calls
> `nest_asyncio.apply()`). Every method also exists on the synchronous `WorkflowClient`.

In [ ]:
# This notebook lives one level below notebooks/, where _bootstrap.py lives. Walk up from the
# working dir to find it, add that dir to sys.path, then import it so `import interactly` works
# with no install needed. (Works whether the kernel cwd is notebooks/ or wf_example_notebooks/.)
import os, sys
from pathlib import Path
import nest_asyncio
from dotenv import load_dotenv

nest_asyncio.apply()

_notebooks_dir = None
for _cand in [Path.cwd(), *Path.cwd().parents]:
    if (_cand / '_bootstrap.py').exists():
        _notebooks_dir = _cand
        break
if _notebooks_dir and str(_notebooks_dir) not in sys.path:
    sys.path.insert(0, str(_notebooks_dir))

import _bootstrap  # noqa: F401 - enables `import interactly` (no install needed)

project_root = Path.cwd()
while project_root.parent != project_root:
    if (project_root / '.env').exists():
        break
    project_root = project_root.parent
env_path = project_root / '.env'
if env_path.exists():
    load_dotenv(dotenv_path=str(env_path), override=True)
    print(f'Loaded .env from: {env_path}')
else:
    print(f'Warning: .env file not found at {env_path}')

In [ ]:
# Interactly credentials are read from environment variables.
# Convenience defaults point at the dev "Workflow Illustrations" org; your shell env always wins.
os.environ.setdefault('INTERACTLY_BASE_URL', 'https://api-dev.interactly.ai/workflows')
os.environ.setdefault('INTERACTLY_TEAM_ID', '67458e762b7d3dc15aaea5b5')
os.environ.setdefault('INTERACTLY_USER_ID', '687b1a4f745c8e6806c98d91')

# The bearer token is a secret — never hardcode it in the notebook.
assert os.environ.get('INTERACTLY_API_KEY'), (
    'Set INTERACTLY_API_KEY in your environment before running this notebook.'
)
print(f"TEAM ID is: {os.getenv('INTERACTLY_TEAM_ID')}")
print(f"BASE URL is: {os.getenv('INTERACTLY_BASE_URL')}")

In [ ]:
import nest_asyncio
from interactly import AsyncWorkflowClient

nest_asyncio.apply()
client = AsyncWorkflowClient()
print('Connected to', client._base_url)

## 1. Build the workflow

Import `build_assistant_workflow()` from the example script: a start node, a **Pause Workflow** node whose reason
names the run's dynamic variables, and an end node.

In [ ]:
# Import the builder straight from the example script instead of duplicating it. Importing the module only defines
# functions; its run() is guarded by `if __name__ == "__main__"`, so nothing runs on import.
_wf_examples = None
for _cand in [Path.cwd(), *Path.cwd().parents]:
    if (_cand / 'wf_examples' / 'wf_example_progression_19.py').exists():
        _wf_examples = _cand / 'wf_examples'
        break
assert _wf_examples, 'Could not locate the wf_examples/ directory.'
if str(_wf_examples) not in sys.path:
    sys.path.insert(0, str(_wf_examples))

from wf_example_progression_19 import (
    DYNAMIC_VARIABLES,
    FINISHED,
    build_assistant_workflow,
    control_nodes_refused,
    wait_for_status,
)

workflow_config_full = build_assistant_workflow()
for node in workflow_config_full.node_configs:
    print(f'{node.name:18} {node.type}')

## 2. Upload it

A server that does not enable the Pause Workflow node refuses the workflow with a 400 that names the flag.

In [ ]:
from interactly import BadRequestError

WF_ID = None
try:
    WF_ID = (await client.workflows.create_from_config(workflow_config_full)).id
    print(f'Uploaded workflow id={WF_ID}')
except BadRequestError as error:
    if not control_nodes_refused(error):
        raise
    print(f'Skipped: this server does not enable the Pause Workflow node.\n  {error}')

## 3. Start a background run, and wait for it to pause

`start_background_run()` answers once the run exists; its first turn runs in the background and stops at the
approval step, so the run is soon `paused`.

In [ ]:
RUN_ID = None
if WF_ID:
    started = await client.workflows.start_background_run(
        WF_ID, dynamic_variables=DYNAMIC_VARIABLES, run_by='Example 19 notebook'
    )
    RUN_ID = started.workflow_run_id
    print(f'Started background run {RUN_ID} ({started.status})')

    run = await wait_for_status(client, RUN_ID, {'paused'} | FINISHED)
    print(f'Run is {run.status}; durable={run.durable}')
    print('Waiting on:', run.pause_summary)

## 4. Find the step waiting for a person

`list_pauses(open_only=True)` lists what the run waits on. `waits_for_person` marks the steps an approval names;
`wake["reason"]` is what the approver sees.

In [ ]:
approval = None
if RUN_ID:
    pauses = await (await client.runs.list_pauses(RUN_ID, open_only=True)).list_all()
    for pause in pauses:
        print(f"{pause.pause_id}  {pause.wake.get('kind'):8}  {pause.state:6}  {pause.wake.get('reason')!r}")
    approval = next(pause for pause in pauses if pause.waits_for_person)

## 5. Approve it

`resume(pause_ids=...)` approves the named steps, and only them, with a note and data for the node. (Resuming without
`pause_ids` would lift a *person's* pause of the run and leave this step waiting.)

In [ ]:
if approval:
    result = await client.runs.resume(
        RUN_ID, pause_ids=[approval.pause_id], note='Approved for 30 days', data={'approved': True, 'days': 30}
    )
    print(f'Approved: {result.outcome}; made ready: {result.made_ready}')

    run = await wait_for_status(client, RUN_ID, FINISHED)
    print(f'Run finished: {run.status}')
    assert run.status == 'completed', run.status

## 6. Cleanup

Delete the workflow (removing its versions and this run) so nothing is left on the server.

In [ ]:
if WF_ID:
    await client.workflows.delete(WF_ID)
    print(f'Workflow {WF_ID} deleted.')

await client.close()
print('Client closed.')